# YOLO11n-InceptionDW: AMP-stable training-only P2 Gaussian supervision

This notebook runs only the corrected P2 auxiliary experiment. The earlier
`yolo11n_inceptiondw_p2_gaussian_aux_640` run is invalid because its auxiliary
loss became `inf`/`nan`; this notebook never resumes that checkpoint.

The corrected focal calculation is evaluated in FP32 with stable
`logsigmoid` identities. Model forward and native YOLO losses still use AMP.
Run cells in order. Formal training runs directly in the notebook kernel.


In [ ]:
# Cell 1 — mount Drive and declare immutable experiment settings
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
SOURCE_DATA_ROOT = DRIVE_ROOT / "ship_detection" / "data"
DRIVE_RUNS = DRIVE_ROOT / "ship_detection" / "runs_spddown_p2aux"
DRIVE_REPORTS = DRIVE_ROOT / "ship_detection" / "preflight_spddown_p2aux"

REPO_URL = "https://github.com/HoverdZ/ship-yolo.git"
BRANCH = "feature/spddown-p2-gaussian-aux"
REPO = Path("/content/ship-yolo")
LOCAL_DATA_ROOT = Path("/content/datasets/ship_detection")

for path in (DRIVE_RUNS, DRIVE_REPORTS):
    path.mkdir(parents=True, exist_ok=True)
print("Drive dataset:", SOURCE_DATA_ROOT)
print("Drive runs:", DRIVE_RUNS)


In [ ]:
# Cell 2 — securely clone/update the private branch, then pin its commit
import getpass
import os
import stat
import subprocess

token = getpass.getpass("GitHub token (input hidden; never written to notebook): ").strip()
if not token:
    raise ValueError("GitHub token cannot be empty.")

askpass = Path("/content/.ship_yolo_askpass.py")
askpass.write_text(
    "#!/usr/bin/env python3\n"
    "import os, sys\n"
    "prompt = sys.argv[1] if len(sys.argv) > 1 else ''\n"
    "print('x-access-token' if 'Username' in prompt else os.environ['SHIP_GITHUB_TOKEN'])\n",
    encoding="utf-8",
)
askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
env = {
    **os.environ,
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
    "SHIP_GITHUB_TOKEN": token,
}

def run_git(arguments, *, authenticated=False):
    completed = subprocess.run(
        ["git", *arguments],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=env if authenticated else None,
    )
    output = completed.stdout + completed.stderr
    if token:
        output = output.replace(token, "[REDACTED]")
    output = output.strip()
    if completed.returncode:
        command = "git " + " ".join(str(part) for part in arguments)
        raise RuntimeError(
            f"{command} failed with exit status {completed.returncode}.\n"
            f"{output or 'Git returned no diagnostic output.'}"
        )
    if output:
        print(output)
    return completed.stdout.strip()

try:
    if REPO.exists():
        if not (REPO / ".git").is_dir():
            if any(REPO.iterdir()):
                raise RuntimeError(
                    f"{REPO} exists but is not a Git clone. "
                    "Move or remove that unrelated directory, then rerun Cell 2."
                )
            REPO.rmdir()

    if REPO.exists():
        remote = run_git(["-C", str(REPO), "remote", "get-url", "origin"])
        if remote.rstrip("/") not in {REPO_URL.rstrip("/"), REPO_URL.removesuffix(".git")}:
            raise RuntimeError(f"Refusing unrelated existing repository: {remote}")
        run_git(
            ["-C", str(REPO), "fetch", "origin", BRANCH],
            authenticated=True,
        )
        run_git(["-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"])
    else:
        run_git(
            [
                "-c", "credential.helper=", "clone", "--branch", BRANCH,
                "--single-branch", REPO_URL, str(REPO),
            ],
            authenticated=True,
        )
finally:
    token = ""
    env.pop("SHIP_GITHUB_TOKEN", None)
    os.environ.pop("SHIP_GITHUB_TOKEN", None)
    askpass.unlink(missing_ok=True)

PINNED_COMMIT = run_git(["-C", str(REPO), "rev-parse", "HEAD"])
run_git(["-C", str(REPO), "checkout", "--detach", PINNED_COMMIT])
print("Pinned commit:", PINNED_COMMIT)


In [ ]:
# Cell 3 — install and verify the pinned runtime
%pip install -q ultralytics==8.4.92

import sys
sys.path.insert(0, str(REPO))

import torch
import ultralytics
from custom_modules.register import register_custom_modules

assert ultralytics.__version__ == "8.4.92", ultralytics.__version__
register_custom_modules()
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cell 4 — copy Drive data to local disk with threaded shutil.copyfile
from concurrent.futures import ThreadPoolExecutor
import shutil
import yaml

if not SOURCE_DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Drive dataset directory not found: {SOURCE_DATA_ROOT}")

if LOCAL_DATA_ROOT.exists():
    shutil.rmtree(LOCAL_DATA_ROOT)
LOCAL_DATA_ROOT.mkdir(parents=True)

source_files = [
    path for path in SOURCE_DATA_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() != ".cache"
]
if not source_files:
    raise RuntimeError(f"No files found under {SOURCE_DATA_ROOT}")

def copy_one(source: Path):
    relative = source.relative_to(SOURCE_DATA_ROOT)
    destination = LOCAL_DATA_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, destination)
    return relative

with ThreadPoolExecutor(max_workers=min(32, (os.cpu_count() or 4) * 2)) as pool:
    copied = list(pool.map(copy_one, source_files))
print(f"Copied {len(copied)} files with shutil.copyfile")

image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
if (LOCAL_DATA_ROOT / "train" / "images").is_dir():
    split_paths = {split: f"{split}/images" for split in ("train", "val", "test")}
elif (LOCAL_DATA_ROOT / "images" / "train").is_dir():
    split_paths = {split: f"images/{split}" for split in ("train", "val", "test")}
else:
    raise RuntimeError("Expected train/images or images/train dataset layout.")

counts = {}
for split, relative in split_paths.items():
    folder = LOCAL_DATA_ROOT / relative
    counts[split] = sum(
        path.is_file() and path.suffix.lower() in image_suffixes for path in folder.rglob("*")
    )
print("Cloud dataset copied successfully; local image counts:", counts)

LOCAL_DATA_YAML = LOCAL_DATA_ROOT / "data.yaml"
LOCAL_DATA_YAML.write_text(
    yaml.safe_dump(
        {
            "path": str(LOCAL_DATA_ROOT),
            "train": split_paths["train"],
            "val": split_paths["val"],
            "test": split_paths["test"],
            "nc": 1,
            "names": {0: "ship"},
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)
print(LOCAL_DATA_YAML.read_text())


In [ ]:
# Cell 5 — verify the AMP-stable loss and run the P2-only 640 preflight
import inspect
import torch

from custom_modules.p2_gaussian_aux import dense_gaussian_focal_loss
from tools.check_spddown_p2aux import main as run_preflight

loss_source = inspect.getsource(dense_gaussian_focal_loss)
assert "logsigmoid" in loss_source
assert "logits.float()" in loss_source

extreme_logits = torch.tensor(
    [[[[-100.0, -20.0, -10.0, 0.0, 10.0, 20.0, 100.0]]]],
    dtype=torch.float16,
    requires_grad=True,
)
extreme_target = torch.tensor(
    [[[[1.0, 0.75, 0.25, 0.0, 0.0, 0.0, 0.0]]]],
    dtype=torch.float16,
)
stability_loss = dense_gaussian_focal_loss(extreme_logits, extreme_target)
stability_loss.backward()
assert stability_loss.dtype == torch.float32
assert torch.isfinite(stability_loss)
assert extreme_logits.grad is not None and torch.isfinite(extreme_logits.grad).all()
print("AMP stability check passed; loss:", float(stability_loss.detach()))

reports = run_preflight(
    [
        "--variant", "p2_gaussian_aux",
        "--weights", str(REPO / "yolo11n.pt"),
        "--imgsz", "640",
        "--output-dir", str(DRIVE_REPORTS / PINNED_COMMIT / "ampstable"),
    ]
)
assert reports["p2_gaussian_aux"]["all_checks_passed"]
print("P2 Gaussian auxiliary 640 preflight passed.")


## Formal corrected experiment

This starts a clean run named `yolo11n_inceptiondw_p2_gaussian_aux_ampstable_640`. It does not read the invalid
`yolo11n_inceptiondw_p2_gaussian_aux_640/weights/last.pt` checkpoint. Do not change the new name
back to the invalid run name.


In [ ]:
# Cell 6 — START CLEAN FORMAL P2 GAUSSIAN AUXILIARY TRAINING
from tools.train_spddown_p2aux import TrainingRequest, run_training

RUN_NAME = "yolo11n_inceptiondw_p2_gaussian_aux_ampstable_640"
INVALID_RUN_NAME = "yolo11n_inceptiondw_p2_gaussian_aux_640"
invalid_checkpoint = DRIVE_RUNS / INVALID_RUN_NAME / "weights" / "last.pt"
print("Invalid checkpoint intentionally ignored:", invalid_checkpoint)

p2_request = TrainingRequest(
    variant="p2_gaussian_aux",
    data=str(LOCAL_DATA_YAML),
    project=str(DRIVE_RUNS),
    weights=str(REPO / "yolo11n.pt"),
    name=RUN_NAME,
    epochs=150,
    imgsz=640,
    batch=8,
    workers=2,
    device="0",
    seed=0,
    optimizer="auto",
    resume=False,
)
p2_result = run_training(p2_request)


In [ ]:
# Cell 7 — resume only the corrected run after an interruption
from tools.train_spddown_p2aux import TrainingRequest, run_training

RUN_NAME = "yolo11n_inceptiondw_p2_gaussian_aux_ampstable_640"
last_pt = DRIVE_RUNS / RUN_NAME / "weights" / "last.pt"
if not last_pt.is_file():
    raise FileNotFoundError(
        f"No corrected checkpoint found: {last_pt}. "
        "Never substitute the invalid p2_gaussian_aux_640 checkpoint."
    )
print("Resuming corrected checkpoint:", last_pt)

resume_request = TrainingRequest(
    variant="p2_gaussian_aux",
    data=str(LOCAL_DATA_YAML),
    project=str(DRIVE_RUNS),
    weights=str(REPO / "yolo11n.pt"),
    name=RUN_NAME,
    resume=True,
)
resume_result = run_training(resume_request)


In [ ]:
# Cell 8 — inspect metrics and reject any non-finite auxiliary loss
import numpy as np
import pandas as pd

RUN_NAME = "yolo11n_inceptiondw_p2_gaussian_aux_ampstable_640"
csv_path = DRIVE_RUNS / RUN_NAME / "results.csv"
if not csv_path.is_file():
    raise FileNotFoundError(csv_path)

frame = pd.read_csv(csv_path)
aux_column = "train/p2_aux_loss"
if aux_column not in frame:
    raise KeyError(f"Missing {aux_column} in {csv_path}")
if not np.isfinite(frame[aux_column].to_numpy(dtype=float)).all():
    raise RuntimeError("Non-finite P2 auxiliary loss detected; stop this run.")

print("Recorded epochs:", len(frame))
print("All recorded P2 auxiliary losses are finite.")
display(
    frame[
        [
            "epoch",
            aux_column,
            "metrics/precision(B)",
            "metrics/recall(B)",
            "metrics/mAP50(B)",
            "metrics/mAP50-95(B)",
        ]
    ].tail(20)
)
